# Model agreement / disagreement across the 52 weeks

Compares the three EV-detection models (XGBoost, Logistic Regression, Random Forest)
using the updated weekly summary files in `../data`.

Two figures are produced (saved to `../results`):

1. **Mean EV charger probability per week** — raw series (level agreement / offset) and
   the same series standardised per model (shape agreement).
2. **Share of meters flagged per probability threshold** — small multiples, one panel
   per threshold, showing where the models agree (moderate thresholds) and where they
   diverge (the high-confidence tail).

Note: these are population-level aggregates — they show agreement in counts and levels,
not whether the models flag the *same* meters. Per-meter agreement is computed in the
private DST environment and reported separately as a table.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
DATA_DIR    = Path("../data")
RESULTS_DIR = Path("../results")

MODEL_FILES = {
    "XGBoost":             "XGB_res_weekly_summary.csv",
    "Logistic Regression": "LogReg_res_weekly_summary.csv",
    "Random Forest":       "RF_res_weekly_summary.csv",
}

MODEL_SHORT = {
    "XGBoost":             "XGB",
    "Logistic Regression": "LogReg",
    "Random Forest":       "RF",
}

# Fixed colour per model (never reassigned), plus a distinct linestyle so the
# figures survive greyscale printing.
MODEL_COLORS = {
    "XGBoost":             "#2a78d6",
    "Logistic Regression": "#008300",
    "Random Forest":       "#e87ba4",
}
MODEL_STYLES = {
    "XGBoost":             "-",
    "Logistic Regression": (0, (5, 2)),
    "Random Forest":       (0, (1, 1.2)),
}

PRED_COLS  = ["n_pred_EV_50", "n_pred_EV_70", "n_pred_EV_80",
              "n_pred_EV_90", "n_pred_EV_95", "n_pred_EV_99"]
THRESHOLDS = [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]

In [ ]:
dfs = {m: pl.read_csv(DATA_DIR / f).sort("week_id") for m, f in MODEL_FILES.items()}

weeks     = dfs["XGBoost"]["week_id"].to_list()
x         = np.arange(len(weeks))
mean_prob = {m: df["mean_probability"].to_numpy() for m, df in dfs.items()}

for m, df in dfs.items():
    print(f"{m:>20}: {df.height} weeks, mean prob {mean_prob[m].mean():.4f}")

In [ ]:
def style_axis(ax):
    """Shared cosmetics: recessive grid, no top/right spines."""
    ax.yaxis.grid(True, linewidth=0.4, color="#e1e0d9", zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


def set_week_ticks(ax, every: int = 4):
    tick_pos = list(range(0, len(weeks), every))
    if (len(weeks) - 1) not in tick_pos:
        tick_pos.append(len(weeks) - 1)
    ax.set_xticks(tick_pos)
    ax.set_xticklabels([str(weeks[i]) for i in tick_pos], fontsize=8.5)
    ax.set_xlim(-0.5, len(weeks) - 0.5)

## Figure 1 — weekly mean EV charger probability

Panel (a) shows the raw series: the models agree on the seasonal dynamics but sit at
different levels (XGBoost ≈ 2 pp below the other two). Panel (b) standardises each
series (z-score per model), collapsing the offset so the shape agreement is directly
visible.

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(11, 7.5), sharex=True, gridspec_kw={"hspace": 0.22}
)

for m in MODEL_FILES:
    y = mean_prob[m]
    ax1.plot(x, y, color=MODEL_COLORS[m], linestyle=MODEL_STYLES[m],
             linewidth=2.0, alpha=0.95, label=m, zorder=3)
    z = (y - y.mean()) / y.std(ddof=1)
    ax2.plot(x, z, color=MODEL_COLORS[m], linestyle=MODEL_STYLES[m],
             linewidth=2.0, alpha=0.95, zorder=3)

# ── Panel (a): raw series ────────────────────────────────────────────
ax1.set_ylabel("Mean EV charger probability", fontsize=10, labelpad=6)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
ax1.set_title("(a) Weekly mean EV charger probability per model", fontsize=11, pad=10)
ax1.legend(loc="upper center", fontsize=9, ncol=3,
           framealpha=0.9, edgecolor="#ddd")
style_axis(ax1)

# ── Panel (b): standardised series ────────────────────────────────────
ax2.set_ylabel("Standardised mean EV charger probability\n(z-score per model)",
               fontsize=10, labelpad=6)
ax2.set_xlabel("Week", fontsize=10, labelpad=6)
ax2.set_title("(b) Same series, standardised per model", fontsize=11, pad=10)
style_axis(ax2)
set_week_ticks(ax2)

fig.tight_layout()
for ext in ("png", "pdf"):
    out = RESULTS_DIR / f"model_agreement_mean_probability.{ext}"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    print(f"Saved → {out}")
plt.show()

## Figure 2 — share of meters flagged, per probability threshold

Small multiples: one panel per threshold, three model lines each (share of scored
meters with predicted EV charger probability above the threshold). The models agree
closely at moderate thresholds but diverge in the high-confidence tail — at P ≥ 0.99
Logistic Regression flags roughly twice the share of XGBoost and Random Forest —
i.e. a calibration difference rather than a disagreement about overall EV prevalence.

Note each panel has its own y-scale.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.8), sharex=True,
                         gridspec_kw={"hspace": 0.32, "wspace": 0.28})

for ax, col, thr in zip(axes.flat, PRED_COLS, THRESHOLDS):
    for m, df in dfs.items():
        share = df[col].to_numpy() / df["n_meters_scored"].to_numpy() * 100
        ax.plot(x, share, color=MODEL_COLORS[m], linestyle=MODEL_STYLES[m],
                linewidth=1.8, alpha=0.95, label=m, zorder=3)
    ax.set_title(f"P ≥ {thr:.2f}", fontsize=10, pad=6)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
    ax.tick_params(axis="y", labelsize=8.5)
    style_axis(ax)
    set_week_ticks(ax, every=8)

for ax in axes[:, 0]:
    ax.set_ylabel("Share of meters flagged", fontsize=9.5, labelpad=6)
for ax in axes[1, :]:
    ax.set_xlabel("Week", fontsize=9.5, labelpad=6)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=9.5,
           framealpha=0.9, edgecolor="#ddd", bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Share of meters flagged as likely EV charger, by probability threshold",
             fontsize=12, y=1.07)

fig.tight_layout()
for ext in ("png", "pdf"):
    out = RESULTS_DIR / f"model_agreement_threshold_shares.{ext}"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    print(f"Saved → {out}")
plt.show()

## Summary table — year-average flagged share per threshold

Handy for quoting exact numbers in the thesis text.

In [ ]:
rows = []
for col, thr in zip(PRED_COLS, THRESHOLDS):
    row = {"threshold": f"P ≥ {thr:.2f}"}
    for m, df in dfs.items():
        share = (df[col] / df["n_meters_scored"]).mean() * 100
        row[MODEL_SHORT[m]] = round(share, 2)
    rows.append(row)

summary = pl.DataFrame(rows)
print("Year-average share of meters flagged (%):")
summary